# 🧬 DHGCMDA — Dual-View Heterogeneous Graph Contrastive Learning
## Framework for miRNA-Disease Association Prediction
### Branch: `breakthrough-conformal` — Uncertainty-Aware Type Prediction

**Notebook này chạy toàn bộ pipeline DHGCMDA-fork trên Google Colab**, bao gồm:

| Phần | Nội dung |
|------|----------|
| **Core** | Train model + 5-fold CV → reproduce Top-1 F1 ≈ 0.697 |
| **Ablation** | 5 variants (no_cl, no_hgcn, no_avf, no_hgt, no_dv) |
| **🆕 Conformal** | APS/RAPS/Mondrian prediction sets → coverage guarantee |

### 📌 Kiến trúc
- **Dual-View Feature**: miRNA (Sequence + Functional) × Disease (Gene + Semantic)
- **Hypergraph Neural Network**: KNN + K-means hypergraph construction
- **Cross-Modal Contrastive Learning**: Intra-view + Inter-view contrastive loss
- **Multi-Type Prediction**: 4 types (Circulation, Epigenetics, Target, Genetics)
- **🆕 Conformal Post-hoc**: Distribution-free prediction SETS with coverage guarantee

### 🏆 Best Config (Plan M)
- `predictor_mode = full_bilinear`, `K_neigs = 2`, `exist_weight = 0.1`
- **Headline: Top-1 F1 = 0.697 ± 0.003** (vs paper 0.597, +16.8%)

---

## 📦 Bước 1: Thiết lập môi trường

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🖥️ Kiểm tra GPU / Runtime
# ═══════════════════════════════════════════════════════════════
import torch

print("═" * 60)
print("🖥️  THÔNG TIN RUNTIME GOOGLE COLAB")
print("═" * 60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"   CUDA version: {torch.version.cuda}")
    DEVICE = 'cuda'
else:
    print("⚠️  Không có GPU — sẽ chạy trên CPU (chậm hơn ~10x)")
    print("   💡 Tip: Runtime → Change runtime type → T4 GPU")
    DEVICE = 'cpu'

print(f"   PyTorch: {torch.__version__}")
print(f"   Device sẽ dùng: {DEVICE}")
print("═" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📥 Clone repo — branch breakthrough-conformal
# ═══════════════════════════════════════════════════════════════
import os

REPO_URL = "https://github.com/hoangtien07/DHGCMDA-fork.git"
BRANCH = "breakthrough-conformal"  # ← Branch chính
PROJECT_DIR = "/content/DHGCMDA-fork"

if not os.path.exists(PROJECT_DIR):
    print(f"📥 Đang clone branch '{BRANCH}' từ {REPO_URL}...")
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {PROJECT_DIR}
    print("✅ Clone thành công!")
else:
    print(f"📂 Thư mục {PROJECT_DIR} đã tồn tại.")
    # Đảm bảo đúng branch
    os.chdir(PROJECT_DIR)
    !git checkout {BRANCH} 2>/dev/null || echo "Đã ở branch {BRANCH}"
    !git pull origin {BRANCH} --ff-only 2>/dev/null || echo "Pull skipped"

os.chdir(PROJECT_DIR)
print(f"\n📁 Thư mục: {os.getcwd()}")
print(f"🌿 Branch:")
!git branch --show-current
print(f"\n📄 Files chính:")
!ls -la *.py conformal_type_prediction.py 2>/dev/null | head -25

In [ ]:
# ──────── THAY THẾ: Mount từ Google Drive ────────
# Bỏ comment nếu bạn đã upload lên Drive thay vì clone

# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = '/content/drive/MyDrive/DHGCMDA-fork'  # ← Đổi path
# os.chdir(PROJECT_DIR)
# print(f"📁 Thư mục hiện tại: {os.getcwd()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📦 Cài đặt Dependencies
# ═══════════════════════════════════════════════════════════════
# Colab đã có: torch, numpy, pandas, scikit-learn, scipy, matplotlib
# Cần thêm: torch-geometric, python-docx, openpyxl, xlrd

print("📦 Đang cài đặt dependencies...")
!pip install -q torch-geometric
!pip install -q python-docx openpyxl xlrd

# Verify
import torch, torch_geometric, numpy as np, pandas as pd, sklearn, scipy

print("\n✅ Dependencies sẵn sàng!")
print(f"   PyTorch:      {torch.__version__}")
print(f"   PyG:          {torch_geometric.__version__}")
print(f"   NumPy:        {np.__version__}")
print(f"   SciPy:        {scipy.__version__}")
print(f"   Scikit-learn: {sklearn.__version__}")

---
## ✅ Bước 2: Kiểm tra dữ liệu

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ✅ Kiểm tra dữ liệu HMDD v2.0
# ═══════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd

DATA_DIR = 'v2.0_495m383D'

required_files = [
    'D_SSM1.txt',     # Disease semantic similarity (View 2)
    'D_SSM2.txt',     # Disease gene similarity
    'M_FSM.txt',      # miRNA functional similarity (View 2)
    'M_GSM.txt',      # miRNA sequence similarity (View 1)
    'multi_all_mirna_disease_pairs_without_negative.csv',
]

print("═" * 60)
print("📊 KIỂM TRA DỮ LIỆU HMDD v2.0")
print("═" * 60)

all_ok = True
for f in required_files:
    path = os.path.join(DATA_DIR, f)
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"  ✅ {f:55s} ({size_kb:.0f} KB)")
    else:
        print(f"  ❌ {f:55s} THIẾU!")
        all_ok = False

if all_ok:
    assoc = pd.read_csv(os.path.join(DATA_DIR, 'multi_all_mirna_disease_pairs_without_negative.csv'), header=None)
    m_gsm = np.loadtxt(os.path.join(DATA_DIR, 'M_GSM.txt'))
    d_ssm1 = np.loadtxt(os.path.join(DATA_DIR, 'D_SSM1.txt'))

    n_mirna, n_disease, n_assoc = m_gsm.shape[0], d_ssm1.shape[0], len(assoc)

    print(f"\n📈 Thống kê:")
    print(f"   miRNAs: {n_mirna} | Diseases: {n_disease} | Assoc: {n_assoc} | Density: {n_assoc/(n_mirna*n_disease)*100:.2f}%")

    type_names = {1: 'Circulation', 2: 'Epigenetics', 3: 'Target', 4: 'Genetics'}
    print(f"\n   Phân bố types:")
    for t_id, t_name in type_names.items():
        count = (assoc.iloc[:, 2] == t_id).sum()
        pct = count / n_assoc * 100
        print(f"     Type {t_id} ({t_name:12s}): {count:4d} ({pct:5.1f}%) {'█' * int(pct/2)}")
    print("\n✅ Dữ liệu sẵn sàng!")
else:
    print("\n❌ Thiếu file! Kiểm tra lại thư mục v2.0_495m383D/")

---
## 🚀 Bước 3: Huấn luyện mô hình

| Chế độ | Epochs | Folds | Thời gian (GPU) | Mục đích |
|--------|--------|-------|-----------------|----------|
| 🧪 Quick Test | 10 | 2 | ~2 phút | Kiểm tra pipeline |
| 🏆 Best Config | 650 | 5 | ~15-25 phút | Reproduce headline 0.697 |
| 🔬 Ablation | 650 | 5 | ~2 giờ | 6 variants (Fig. 4 paper) |

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🧪 QUICK TEST — Smoke test (~2 phút GPU)
# ═══════════════════════════════════════════════════════════════

!python main_experiments_hetero1.py \
    --device {DEVICE} \
    --dataset v2.0_495m383D \
    --K_neigs 2 \
    --predictor_mode full_bilinear \
    --exist_weight 0.1 \
    --epoch 10 \
    --validation 2 \
    --seed 1234

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🏆 BEST CONFIG (Plan M — K=2, Full Bilinear)
# Target: Top-1 F1 ≈ 0.697 | ~15-25 phút GPU
# ═══════════════════════════════════════════════════════════════

!mkdir -p logs results
!python main_experiments_hetero1.py \
    --device {DEVICE} \
    --dataset v2.0_495m383D \
    --K_neigs 2 \
    --predictor_mode full_bilinear \
    --exist_weight 0.1 \
    --epoch 650 \
    --validation 5 \
    --seed 1234 \
    2>&1 | tee logs/colab_best_config.log

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🔬 ABLATION STUDY — Baseline + 5 variants (~2 giờ GPU)
# ═══════════════════════════════════════════════════════════════

import subprocess, os, json, time

os.makedirs('logs', exist_ok=True)
os.makedirs('results', exist_ok=True)

BASE_CMD = [
    'python', 'main_experiments_hetero1.py',
    '--device', DEVICE,
    '--dataset', 'v2.0_495m383D',
    '--K_neigs', '2',
    '--predictor_mode', 'full_bilinear',
    '--exist_weight', '0.1',
    '--epoch', '650',
    '--validation', '5',
    '--seed', '1234',
]

VARIANTS = [
    ('baseline', 'none'),
    ('no_cl',    'no_cl'),     # w/o Contrastive Learning
    ('no_hgcn',  'no_hgcn'),   # w/o Hypergraph Conv
    ('no_avf',   'no_avf'),    # w/o Attention View Fusion
    ('no_hgt',   'no_hgt'),    # w/o HGT layers
    ('no_dv',    'no_dv'),     # w/o Dual-View
]

results = {}
total_start = time.time()

for name, ablation in VARIANTS:
    print(f"\n{'═' * 70}")
    print(f"🔬 Running: {name} (ablation={ablation})")
    print(f"{'═' * 70}")

    cmd = BASE_CMD + ['--ablation', ablation]
    log_file = f'logs/colab_ablation_{name}.log'

    start = time.time()
    result = subprocess.run(
        cmd, capture_output=True, text=True,
        env={**os.environ, 'PYTHONUTF8': '1'}
    )
    elapsed = time.time() - start

    with open(log_file, 'w', encoding='utf-8') as f:
        f.write(result.stdout)
        if result.stderr:
            f.write('\n--- STDERR ---\n' + result.stderr)

    # Parse metrics
    metrics = {'time_seconds': elapsed}
    for line in result.stdout.split('\n'):
        line = line.strip()
        if 'Top-1 F1:' in line and 'Metrics' not in line:
            try: metrics['top1_f1'] = float(line.split('Top-1 F1:')[1].strip().split()[0])
            except: pass
        if 'AUC:' in line and 'Binary' not in line and 'CV_type' not in line:
            try:
                val = float(line.split('AUC:')[1].strip().split()[0])
                if val > 0.5: metrics['auc'] = val
            except: pass

    results[name] = metrics
    f1 = metrics.get('top1_f1', 'N/A')
    auc = metrics.get('auc', 'N/A')
    print(f"  ⏱️ {elapsed:.0f}s | Top-1 F1: {f1} | AUC: {auc}")

with open('results/colab_ablation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

total_elapsed = time.time() - total_start
print(f"\n{'═' * 70}")
print(f"✅ Ablation hoàn tất! Tổng: {total_elapsed/60:.1f} phút")
print(f"📄 Saved: results/colab_ablation_results.json")

---
## 🆕 Bước 4: Conformal Type Prediction (Breakthrough Feature)

**Ý tưởng**: Thay vì chỉ dự đoán 1 type (Top-1), sinh **tập dự đoán (prediction set)** với **đảm bảo coverage ≥ 1−α**.

| Phương pháp | Mô tả |
|-------------|--------|
| **APS** | Adaptive Prediction Sets — marginal coverage guarantee |
| **RAPS** | Regularized APS — nhỏ hơn set-size cho large C |
| **Mondrian** | Class-conditional — coverage guarantee **per-class** (quan trọng cho minority types) |

Pipeline: `Train + dump scores → conformal_type_prediction.py → coverage report`

### 4a. v2.0 (4-type)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🧪 STEP 1: Train best config + DUMP per-fold held-out scores
# Flag --dump_scores → lưu type_probs, true_type cho mỗi fold
# ═══════════════════════════════════════════════════════════════

!mkdir -p results/conformal/v2_dump logs

!python main_experiments_hetero1.py \
    --device {DEVICE} \
    --dataset v2.0_495m383D \
    --predictor_mode full_bilinear \
    --K_neigs 2 \
    --exist_weight 0.1 \
    --seed 1234 \
    --epoch 650 \
    --validation 5 \
    --dump_scores results/conformal/v2_dump/ \
    2>&1 | tee logs/conformal_v2_dump.log

# Verify dump files
import glob, os
dumps = sorted(glob.glob('results/conformal/v2_dump/fold*.npz'))
print(f"\n✅ Dumped {len(dumps)} fold files:")
for f in dumps:
    print(f"   📄 {f} ({os.path.getsize(f)/1024:.1f} KB)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📊 STEP 2: Conformal Analysis — APS / RAPS / Mondrian
# Post-hoc, KHÔNG train lại model, chỉ dùng numpy
# ═══════════════════════════════════════════════════════════════

!python conformal_type_prediction.py \
    --dump_dir results/conformal/v2_dump/ \
    --alpha 0.1 0.05 \
    --out results/conformal/v2_conformal_report.json

print("\n" + "═" * 60)
print("📊 CONFORMAL REPORT — v2.0 (4-type)")
print("═" * 60)

# Load và hiển thị đẹp
import json
with open('results/conformal/v2_conformal_report.json') as f:
    report = json.load(f)

print(f"   Model Top-1 accuracy: {report['top1_accuracy']:.4f}")
print(f"   Positive pairs:       {report['n_total']}")
print(f"   Number of types:      {report['num_types']}")

type_names = {1: 'Circulation', 2: 'Epigenetics', 3: 'Target', 4: 'Genetics', 5: 'Tissue'}

for r in report['results']:
    alpha = r['alpha']
    target = r['target']
    print(f"\n{'─' * 60}")
    print(f"  α = {alpha} → target coverage = {target:.0%}")
    print(f"{'─' * 60}")

    for method in ['APS', 'RAPS', 'Mondrian']:
        m = r[method]
        cov = m['marginal_coverage']
        sz = m['avg_set_size']
        status = '✅' if cov >= target else '⚠️'
        print(f"  {method:10s}: coverage = {cov:.4f} {status}  |  set-size = {sz:.3f}/{report['num_types']}")

    # Per-class for APS
    print(f"\n  Per-class coverage (APS):")
    for k, v in r['APS']['per_class'].items():
        name = type_names.get(int(k), f'T{k}')
        print(f"    {name:12s}: {v['coverage']:.3f} (n={v['n']})")

    # Shuffle control
    ctrl = r['shuffle_control']
    print(f"\n  🔀 Negative control (shuffle): coverage={ctrl['marginal_coverage']:.4f}, size={ctrl['avg_set_size']:.3f}")
    print(f"     → APS set-size {r['APS']['avg_set_size']:.2f} vs shuffle {ctrl['avg_set_size']:.2f} = model cắt ~{(ctrl['avg_set_size'] - r['APS']['avg_set_size']):.1f} lớp")

### 4b. v3.2 (5-type, bao gồm Tissue) — Tuỳ chọn

Chạy conformal trên v3.2 để thấy:
- Metric-bug xác nhận (official Top-1 F1 = 0.0 nhưng acc thật ≈ 0.30)
- APS che giấu collapse type Tissue (T5 = 0.64) → **Mondrian phục hồi (T5 = 0.96)**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🧬 Conformal v3.2 (5-type) — optional, ~75 phút GPU
# ═══════════════════════════════════════════════════════════════

# Kiểm tra v3.2_wang data có sẵn không
import os
v32_exists = os.path.exists('v3.2_wang') and os.path.isdir('v3.2_wang')

if v32_exists:
    print("📂 v3.2_wang dataset found! Chạy conformal v3.2...")

    !mkdir -p results/conformal/v32_dump

    # Train + dump (300ep, 5fold)
    !python main_experiments_hetero1.py \
        --device {DEVICE} \
        --dataset v3.2_wang \
        --predictor_mode full_bilinear \
        --exist_weight 0.1 \
        --loss_mode two_head \
        --epoch 300 \
        --validation 5 \
        --dump_scores results/conformal/v32_dump/ \
        2>&1 | tee logs/conformal_v32_dump.log

    # Conformal analysis
    !python conformal_type_prediction.py \
        --dump_dir results/conformal/v32_dump/ \
        --alpha 0.1 0.05 \
        --out results/conformal/v32_conformal_report.json

    print("\n✅ v3.2 conformal analysis hoàn tất!")
    print("📄 Report: results/conformal/v32_conformal_report.json")
else:
    print("⚠️ v3.2_wang dataset không có sẵn trong repo.")
    print("   Bỏ qua conformal v3.2 — chỉ chạy v2.0.")
    print("   (Để có v3.2: chạy preprocess_v32_wang.py với dữ liệu HMDD v3.2 raw)")

---
## 🎲 Bước 5 (Tuỳ chọn): Multi-Seed Verification

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🎲 Multi-Seed — 3 seeds × best config
# ═══════════════════════════════════════════════════════════════

import subprocess, json, time, os
import numpy as np

os.makedirs('logs', exist_ok=True)
os.makedirs('results', exist_ok=True)

SEEDS = [1234, 0, 42]
seed_results = {}

for seed in SEEDS:
    print(f"\n{'═' * 60}")
    print(f"🎲 Seed = {seed}")
    print(f"{'═' * 60}")

    cmd = [
        'python', 'main_experiments_hetero1.py',
        '--device', DEVICE,
        '--dataset', 'v2.0_495m383D',
        '--K_neigs', '2',
        '--predictor_mode', 'full_bilinear',
        '--exist_weight', '0.1',
        '--epoch', '650',
        '--validation', '5',
        '--seed', str(seed),
    ]

    start = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True,
                           env={**os.environ, 'PYTHONUTF8': '1'})
    elapsed = time.time() - start

    with open(f'logs/colab_seed_{seed}.log', 'w', encoding='utf-8') as f:
        f.write(result.stdout)

    metrics = {'seed': seed, 'time_seconds': elapsed}
    for line in result.stdout.split('\n'):
        line = line.strip()
        if 'Top-1 F1:' in line and 'Metrics' not in line:
            try: metrics['top1_f1'] = float(line.split('Top-1 F1:')[1].strip().split()[0])
            except: pass
        if 'AUC:' in line and 'Binary' not in line and 'CV_type' not in line:
            try:
                val = float(line.split('AUC:')[1].strip().split()[0])
                if val > 0.5: metrics['auc'] = val
            except: pass

    seed_results[seed] = metrics
    print(f"  ⏱️ {elapsed:.0f}s | F1: {metrics.get('top1_f1', 'N/A')} | AUC: {metrics.get('auc', 'N/A')}")

# Summary
f1s = [v['top1_f1'] for v in seed_results.values() if 'top1_f1' in v]
aucs = [v['auc'] for v in seed_results.values() if 'auc' in v]

print(f"\n{'═' * 60}")
print(f"📊 MULTI-SEED SUMMARY")
print(f"{'═' * 60}")
print(f"{'Seed':>6} {'Top-1 F1':>10} {'AUC':>10}")
print(f"{'─' * 30}")
for seed, m in seed_results.items():
    print(f"{seed:>6} {m.get('top1_f1', 0):>10.4f} {m.get('auc', 0):>10.4f}")
print(f"{'─' * 30}")
if f1s:
    print(f"{'Mean':>6} {np.mean(f1s):>10.4f} {np.mean(aucs):>10.4f}")
    print(f"{'Std':>6} {np.std(f1s):>10.4f} {np.std(aucs):>10.4f}")
    print(f"\n🏆 Top-1 F1 = {np.mean(f1s):.3f} ± {np.std(f1s):.3f} (paper: 0.597, Δ = +{(np.mean(f1s)-0.597)*100:.1f}%)")

with open('results/colab_multiseed_results.json', 'w') as f:
    json.dump(seed_results, f, indent=2, default=str)

---
## 🔍 Bước 6: Chạy tương tác (Debug / Step-by-step)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🔍 Interactive Setup — Import + Load Data
# ═══════════════════════════════════════════════════════════════
import sys, os, warnings
warnings.filterwarnings('ignore')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import torch, torch.optim as optim, numpy as np, time

from param import parameter_parser
from prepareData import prepare_data
from trainData import Dataset
from hetero_model import HeterogenousGraphCLAMIR
from main_experiments_hetero1 import (
    SimplifiedMultiTypeAssociationLoss,
    create_hetero_data_optimized,
    constructHW_knn,
    seed_torch,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Modules loaded | Device: {device}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🔧 Khởi tạo model + data
# ═══════════════════════════════════════════════════════════════
sys.argv = ['']
args = parameter_parser()

# Best Config (Plan M)
args.device = str(device)
args.K_neigs = [2]
args.predictor_mode = 'full_bilinear'
args.exist_weight = 0.1
args.dataset = 'v2.0_495m383D'
args.epoch = 100        # Rút ngắn cho demo (đổi 650 cho full)
args.validation = 5
args.seed = 1234

seed_torch(args.seed)

print("📊 Đang tải dữ liệu...")
dataset = prepare_data(args)
train_data_obj = Dataset(args, dataset)

print(f"✅ miRNAs: {args.mi_num} | Diseases: {args.dis_num} | Folds: {args.validation} | Epochs: {args.epoch}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🏋️ Train 1 fold với theo dõi chi tiết
# ═══════════════════════════════════════════════════════════════
from main_experiments_hetero1 import (
    train_epoch_optimized,
    evaluate_optimized_with_comprehensive_metrics,
)

fold_idx = 0

hidden_list = [256, 256]
if args.n_head > 0:
    hidden_list = [dim - (dim % args.n_head) for dim in hidden_list]

model = HeterogenousGraphCLAMIR(
    args.mi_num, args.dis_num, hidden_list, 64, args
).to(device)

for param in model.parameters():
    param.data = param.data.float()

optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-5)

print(f"🏋️ Training Fold {fold_idx + 1}...")
start = time.time()
fold_data = train_data_obj[fold_idx]
true_one, true_zero, pre_one, pre_zero = train_epoch_optimized(
    model, fold_data, optimizer, args
)
elapsed = time.time() - start

result = evaluate_optimized_with_comprehensive_metrics(
    true_one, true_zero, pre_one, pre_zero
)

if isinstance(result, tuple) and len(result) >= 3:
    binary_metrics, cv_type_metrics, top1_metrics = result[:3]
    print(f"\n📊 Fold {fold_idx+1} ({elapsed:.0f}s):")
    print(f"   AUC: {binary_metrics[0][0]:.4f} | AUPR: {binary_metrics[0][1]:.4f}")
    print(f"   Binary F1: {binary_metrics[0][2]:.4f} | Top-1 F1: {top1_metrics['top1_f1']:.4f}")

---
## 📊 Bước 7: Visualization kết quả

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📊 Conformal Visualization — Coverage & Set-size
# ═══════════════════════════════════════════════════════════════
import json, os
import matplotlib.pyplot as plt
import numpy as np

REPORT_FILE = 'results/conformal/v2_conformal_report.json'

if os.path.exists(REPORT_FILE):
    with open(REPORT_FILE) as f:
        report = json.load(f)

    C = report['num_types']
    type_names_map = {1: 'Circulation', 2: 'Epigenetics', 3: 'Target', 4: 'Genetics', 5: 'Tissue'}

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ── Plot 1: Method comparison (coverage) ──
    ax = axes[0]
    methods = ['APS', 'RAPS', 'Mondrian']
    colors = ['#4A90D9', '#D94A4A', '#4AD977']

    for r in report['results']:
        alpha = r['alpha']
        target = r['target']
        coverages = [r[m]['marginal_coverage'] for m in methods]
        set_sizes = [r[m]['avg_set_size'] for m in methods]

        x = np.arange(len(methods))
        bars = ax.bar(x, coverages, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
        ax.axhline(y=target, color='red', linestyle='--', alpha=0.7, label=f'Target {target:.0%}')

        for bar, val in zip(bars, coverages):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

    ax.set_xticks(x)
    ax.set_xticklabels(methods, fontsize=12)
    ax.set_ylabel('Marginal Coverage', fontsize=12)
    ax.set_title(f'Coverage Guarantee (α={report["results"][0]["alpha"]})', fontsize=14, fontweight='bold')
    ax.set_ylim(0.8, 1.02)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)

    # ── Plot 2: Per-class coverage (APS vs Mondrian) ──
    ax = axes[1]
    r = report['results'][0]  # α=0.1
    aps_pc = r['APS']['per_class']
    mond_pc = r['Mondrian']['per_class']

    classes = sorted(aps_pc.keys(), key=int)
    class_labels = [type_names_map.get(int(c), f'T{c}') for c in classes]
    aps_covs = [aps_pc[c]['coverage'] for c in classes]
    mond_covs = [mond_pc[c]['coverage'] for c in classes]

    x = np.arange(len(classes))
    w = 0.35
    bars1 = ax.bar(x - w/2, aps_covs, w, label='APS', color='#4A90D9', alpha=0.85)
    bars2 = ax.bar(x + w/2, mond_covs, w, label='Mondrian', color='#4AD977', alpha=0.85)
    ax.axhline(y=0.9, color='red', linestyle='--', alpha=0.7, label='Target 90%')

    for bar, val in zip(bars1, aps_covs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)
    for bar, val in zip(bars2, mond_covs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(class_labels, fontsize=10, rotation=20, ha='right')
    ax.set_ylabel('Per-class Coverage', fontsize=12)
    ax.set_title('Per-class: APS vs Mondrian', fontsize=14, fontweight='bold')
    ax.set_ylim(0.6, 1.05)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)

    # ── Plot 3: Set-size comparison (APS vs Shuffle) ──
    ax = axes[2]
    for r in report['results']:
        alpha = r['alpha']
        data = {
            'APS': r['APS']['avg_set_size'],
            'RAPS': r['RAPS']['avg_set_size'],
            'Mondrian': r['Mondrian']['avg_set_size'],
            'Shuffle\n(control)': r['shuffle_control']['avg_set_size'],
        }
        names = list(data.keys())
        vals = list(data.values())
        bar_colors = ['#4A90D9', '#D94A4A', '#4AD977', '#999999']

        bars = ax.bar(names, vals, color=bar_colors, alpha=0.85, edgecolor='white', linewidth=1.5)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{val:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

    ax.axhline(y=C, color='gray', linestyle=':', alpha=0.5, label=f'Max ({C} types)')
    ax.set_ylabel(f'Avg Set Size (/ {C} types)', fontsize=12)
    ax.set_title(f'Set Size — Model Info', fontsize=14, fontweight='bold')
    ax.set_ylim(0, C + 0.5)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)

    fig.suptitle('DHGCMDA Conformal Type Prediction — v2.0 (4-type)',
                 fontsize=16, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.savefig('results/conformal/colab_conformal_charts.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("📊 Saved: results/conformal/colab_conformal_charts.png")
else:
    print("⚠️ Chưa có conformal report. Hãy chạy Bước 4 trước!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📊 Bảng tổng hợp kết quả vs Paper
# ═══════════════════════════════════════════════════════════════
import json, os

paper = {'AUC': 0.9669, 'Top-1 F1': 0.5970}

print("═" * 70)
print("📊 TỔNG HỢP KẾT QUẢ — branch breakthrough-conformal")
print("═" * 70)

# 1. Core metrics
print("\n🏆 Core Metrics (Best Config: K=2, full_bilinear):")
print(f"{'Metric':<15} {'Paper':>10} {'Reproduce':>12} {'Δ':>10}")
print(f"{'─' * 50}")

for rf in ['results/colab_multiseed_results.json', 'results/colab_ablation_results.json']:
    if os.path.exists(rf):
        with open(rf) as f:
            loaded = json.load(f)
        bl = loaded.get('baseline', loaded.get(1234, loaded.get('1234', list(loaded.values())[0])))
        for metric, pval in paper.items():
            key = 'auc' if metric == 'AUC' else 'top1_f1'
            oval = bl.get(key, 0)
            if oval > 0:
                delta = (oval - pval) * 100
                print(f"{metric:<15} {pval:>10.4f} {oval:>12.4f} {delta:>+9.1f}%")
        break
else:
    print("  (Chưa có kết quả — chạy Bước 3 hoặc 5)")

# 2. Conformal summary
print("\n🆕 Conformal Type Prediction:")
for label, path in [('v2.0 (4-type)', 'results/conformal/v2_conformal_report.json'),
                     ('v3.2 (5-type)', 'results/conformal/v32_conformal_report.json')]:
    if os.path.exists(path):
        with open(path) as f:
            rpt = json.load(f)
        r = rpt['results'][0]  # α=0.1
        print(f"\n  {label} (α=0.1, target=90%):")
        print(f"    Model Top-1 acc:    {rpt['top1_accuracy']:.4f}")
        print(f"    APS coverage:       {r['APS']['marginal_coverage']:.4f} ✅")
        print(f"    APS set-size:       {r['APS']['avg_set_size']:.2f}/{rpt['num_types']}")
        print(f"    Mondrian coverage:  {r['Mondrian']['marginal_coverage']:.4f} ✅")
        print(f"    Shuffle set-size:   {r['shuffle_control']['avg_set_size']:.2f} (no info)")

print(f"\n{'═' * 70}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📊 Ablation Bar Chart (nếu đã chạy ablation)
# ═══════════════════════════════════════════════════════════════
import json, os
import matplotlib.pyplot as plt
import numpy as np

RESULTS_FILE = 'results/colab_ablation_results.json'

if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        abl = json.load(f)

    display = {'baseline': 'Full Model', 'no_cl': 'w/o CL', 'no_hgcn': 'w/o HGCN',
               'no_avf': 'w/o AVF', 'no_hgt': 'w/o HGT', 'no_dv': 'w/o DV'}

    names, f1s, aucs = [], [], []
    for key in ['baseline', 'no_cl', 'no_hgcn', 'no_avf', 'no_hgt', 'no_dv']:
        if key in abl:
            names.append(display.get(key, key))
            f1s.append(abl[key].get('top1_f1', 0))
            aucs.append(abl[key].get('auc', 0))

    if names:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        colors = ['#FFB900'] + ['#4A90D9'] * (len(names) - 1)

        bars1 = ax1.bar(names, f1s, color=colors, edgecolor='white', linewidth=1.5)
        ax1.set_ylabel('Top-1 F1', fontsize=12)
        ax1.set_title('Ablation — Top-1 F1', fontsize=14, fontweight='bold')
        ax1.axhline(y=0.597, color='red', linestyle='--', alpha=0.7, label='Paper (0.597)')
        for bar, val in zip(bars1, f1s):
            ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                     f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
        ax1.legend(); ax1.grid(axis='y', alpha=0.3)
        plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right')

        bars2 = ax2.bar(names, aucs, color=colors, edgecolor='white', linewidth=1.5)
        ax2.set_ylabel('AUC', fontsize=12)
        ax2.set_title('Ablation — AUC', fontsize=14, fontweight='bold')
        ax2.axhline(y=0.9669, color='red', linestyle='--', alpha=0.7, label='Paper (0.9669)')
        for bar, val in zip(bars2, aucs):
            ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                     f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
        ax2.set_ylim(0.9, max(aucs)*1.02 if aucs else 1)
        ax2.legend(); ax2.grid(axis='y', alpha=0.3)
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')

        fig.suptitle('DHGCMDA Ablation Study', fontsize=16, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig('results/ablation_chart_colab.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("⚠️ Chưa có ablation results. Chạy Bước 3 (Ablation Study).")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📈 Training Loss Curves
# ═══════════════════════════════════════════════════════════════
import re, os
import matplotlib.pyplot as plt

def parse_log(path):
    epochs, total, recover = [], [], []
    if not os.path.exists(path): return None
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            m = re.search(r'Epoch (\d+), Total Loss: ([\d.]+), Recover Loss: ([\d.]+)', line)
            if m:
                epochs.append(int(m.group(1)))
                total.append(float(m.group(2)))
                recover.append(float(m.group(3)))
    return {'epochs': epochs, 'total': total, 'recover': recover} if epochs else None

for lf in ['logs/conformal_v2_dump.log', 'logs/colab_best_config.log',
           'logs/colab_seed_1234.log', 'logs/colab_ablation_baseline.log']:
    data = parse_log(lf)
    if data:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        ax1.plot(data['epochs'], data['total'], 'b-', linewidth=1.5, alpha=0.8)
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Total Loss')
        ax1.set_title('Total Loss', fontsize=14, fontweight='bold')
        ax1.grid(True, alpha=0.3)

        ax2.plot(data['epochs'], data['recover'], 'r-', linewidth=1.5, alpha=0.8)
        ax2.set_xlabel('Epoch'); ax2.set_ylabel('Recover Loss')
        ax2.set_title('Association Prediction Loss', fontsize=14, fontweight='bold')
        ax2.grid(True, alpha=0.3)

        fig.suptitle(f'Training Progress ({os.path.basename(lf)})',
                     fontsize=16, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig('results/training_curves.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"📈 From: {lf}")
        break
else:
    print("⚠️ Chưa có log file. Chạy training trước.")

---
## 💾 Bước 8: Lưu kết quả về Google Drive

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 💾 Copy results → Google Drive
# ═══════════════════════════════════════════════════════════════
import shutil, os

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')

    SAVE_DIR = '/content/drive/MyDrive/DHGCMDA_Results'
    os.makedirs(SAVE_DIR, exist_ok=True)

    for folder in ['results', 'logs']:
        src, dst = folder, os.path.join(SAVE_DIR, folder)
        if os.path.exists(src):
            if os.path.exists(dst): shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"  ✅ {folder}/ → {dst}")

    print(f"\n💾 Saved: {SAVE_DIR}")
except ImportError:
    print("⚠️ Không phải Colab — bỏ qua.")
except Exception as e:
    print(f"⚠️ Error: {e}")

---
## 💡 Tips & Tham khảo

### ⏱️ Thời gian chạy ước tính
| Task | T4 GPU | CPU |
|------|--------|-----|
| Quick Test (10ep/2fold) | ~2 phút | ~10 phút |
| Best Config (650ep/5fold) | ~15-25 phút | ~4 giờ |
| Ablation (6 variants) | ~2 giờ | ~24 giờ |
| Conformal v2.0 (dump + analysis) | ~20 phút | ~4 giờ |
| Multi-seed (3 seeds) | ~1 giờ | ~12 giờ |

### 🆕 Conformal Type Prediction
- **Post-hoc**: KHÔNG train lại model, chỉ dùng numpy trên dumped scores
- **APS**: Marginal coverage guarantee (≥ 1−α)
- **Mondrian**: Class-conditional → quan trọng cho minority types (Epigenetics)
- **Negative control (shuffle)**: Phân biệt model info vs trivial coverage
- Ref: Romano et al. (2020), "Classification with Valid Adaptive Prediction Sets"

### 🔧 Troubleshooting
1. **CUDA OOM**: Giảm hidden [256,256] → [128,128]
2. **NaN loss**: `--lr 0.00005`
3. **Import error**: Kiểm tra `os.getcwd()` = project dir
4. **v3.2 data**: Cần chạy `preprocess_v32_wang.py` với raw HMDD v3.2

### 📚 Tham khảo
- **Paper**: Sun Y. et al., BMC Bioinformatics 2026
- **Repo**: [CDMBlab/DHGCMDA](https://github.com/CDMBlab/DHGCMDA)
- **Docs**: `CLAUDE.md`, `EXPERIMENT_STATE.md`, `docs/RESULTS_CONFORMAL.md`
- **Branch**: `breakthrough-conformal` — conformal + Plan M (K-sweep)